# Semana 6: La Derivada - Midiendo el Cambio
## Notebook Conceptual (NB1) - Derivadas con Datos Sintéticos

### Propósito de la sesión
Comprender que la **derivada** es la herramienta fundamental que le dice a un modelo cómo ajustar sus parámetros para minimizar el error. Exploraremos el concepto desde su definición geométrica, calcularemos derivadas simbólicamente y las conectaremos con el entrenamiento de modelos de IA.

### Objetivos de aprendizaje
*   Entender la derivada como la pendiente de la recta tangente (tasa de cambio instantánea).
*   Calcular derivadas simbólicas de funciones simples usando **SymPy**.
*   Visualizar la interpretación geométrica de la derivada.
*   Derivar la función de pérdida **MSE** con respecto a un peso.
*   Conectar la derivada con el **gradiente descendente** y el **backpropagation**.

## Configuración Inicial

Importamos las librerías necesarias: SymPy para cálculo simbólico, NumPy para operaciones numéricas y Matplotlib para visualizaciones.

In [ ]:
# Importamos librerías
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from sympy.abc import x, w, b

# Configuración de SymPy
sp.init_printing()

# Configuración de matplotlib
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 5)

print("🎯 Librerías importadas correctamente.")

---
## 1. La Derivada: Idea Intuitiva y Geométrica

La derivada de una función $f(x)$ en un punto $a$ mide la **tasa de cambio instantánea** de la función en ese punto. Geométricamente, es la **pendiente de la recta tangente** a la curva en $(a, f(a))$.

### 1.1. De la Secante a la Tangente

La pendiente de la recta secante entre dos puntos $(x, f(x))$ y $(x+h, f(x+h))$ es:

$$m_{\text{sec}} = \frac{f(x+h) - f(x)}{h}$$

A medida que $h \to 0$, la secante se convierte en la tangente, y obtenemos la derivada:

$$f'(x) = \lim_{h \to 0} \frac{f(x+h) - f(x)}{h}$$

In [ ]:
# Función para visualizar la aproximación de la derivada
def plot_tangent_approximation(f, x0, h_vals=[2, 1, 0.5]):
    x_vals = np.linspace(x0 - 3, x0 + 3, 400)
    y_vals = f(x_vals)
    
    plt.figure(figsize=(12, 6))
    plt.plot(x_vals, y_vals, 'b-', linewidth=2, label='f(x) = x²')
    
    # Punto de interés
    plt.scatter([x0], [f(x0)], color='black', s=100, zorder=5, label='Punto de interés')
    
    # Colores para diferentes h
    colors = ['red', 'orange', 'green']
    
    for i, h in enumerate(h_vals):
        # Punto secundario
        x1 = x0 + h
        # Pendiente de la secante
        m = (f(x1) - f(x0)) / h
        # Recta secante
        y_sec = f(x0) + m * (x_vals - x0)
        plt.plot(x_vals, y_sec, '--', color=colors[i], linewidth=1.5,
                 label=f'Secante (h={h})')
        plt.scatter([x1], [f(x1)], color=colors[i], s=50, zorder=5)
    
    # Recta tangente real (derivada)
    # Para f(x)=x², f'(x)=2x, en x0=1, m=2
    m_tang = 2 * x0
    y_tang = f(x0) + m_tang * (x_vals - x0)
    plt.plot(x_vals, y_tang, 'k-', linewidth=2, label='Tangente (derivada exacta)')
    
    plt.xlabel('x')
    plt.ylabel('f(x)')
    plt.title('Aproximación de la derivada mediante secantes')
    plt.legend()
    plt.grid(True)
    plt.axhline(0, color='gray', linestyle='--', alpha=0.3)
    plt.axvline(0, color='gray', linestyle='--', alpha=0.3)
    plt.show()

# Ejemplo con f(x) = x² en x0 = 1
f_cuad = lambda x: x**2
plot_tangent_approximation(f_cuad, x0=1, h_vals=[2, 1, 0.5])

---
## 2. Cálculo de Derivadas Simbólicas con SymPy

SymPy nos permite calcular derivadas exactas de funciones simbólicas. Veremos ejemplos con funciones simples y de activación.

In [ ]:
# Definimos la variable simbólica
x = sp.Symbol('x')

# 1. f(x) = x²
f1 = x**2
f1_prima = sp.diff(f1, x)
print(f"f(x) = {f1}")
print(f"f'(x) = {f1_prima}\n")

# 2. f(x) = e^x
f2 = sp.exp(x)
f2_prima = sp.diff(f2, x)
print(f"f(x) = {f2}")
print(f"f'(x) = {f2_prima}\n")

# 3. f(x) = ln(x)
f3 = sp.ln(x)
f3_prima = sp.diff(f3, x)
print(f"f(x) = {f3}")
print(f"f'(x) = {f3_prima}\n")

# 4. Función sigmoide
sig = 1 / (1 + sp.exp(-x))
sig_prima = sp.diff(sig, x)
print(f"σ(x) = {sig}")
print(f"σ'(x) = {sp.simplify(sig_prima)}\n")

# 5. Función tanh
tanh_sym = sp.tanh(x)
tanh_prima = sp.diff(tanh_sym, x)
print(f"tanh(x) = {tanh_sym}")
print(f"tanh'(x) = {sp.simplify(tanh_prima)}")

---
## 3. Interpretación Geométrica: Pendiente de la Recta Tangente

La derivada en un punto nos da la pendiente de la recta tangente. Visualicemos esto para diferentes funciones.

In [ ]:
def plot_function_and_tangent(f, f_prime, x0, x_range=(-3, 3), num_points=400):
    """
    Dibuja una función y su recta tangente en x0.
    f: función numérica
    f_prime: función numérica de la derivada
    """
    x_vals = np.linspace(x_range[0], x_range[1], num_points)
    y_vals = f(x_vals)
    
    # Punto de tangencia
    y0 = f(x0)
    m = f_prime(x0)
    
    # Recta tangente: y = y0 + m*(x - x0)
    y_tang = y0 + m * (x_vals - x0)
    
    plt.figure(figsize=(10, 6))
    plt.plot(x_vals, y_vals, 'b-', linewidth=2, label='f(x)')
    plt.plot(x_vals, y_tang, 'r--', linewidth=2, label=f'Tangente en x={x0} (pendiente={m:.2f})')
    plt.scatter([x0], [y0], color='black', s=100, zorder=5)
    plt.xlabel('x')
    plt.ylabel('y')
    plt.title(f'Interpretación geométrica de la derivada')
    plt.legend()
    plt.grid(True)
    plt.axhline(0, color='gray', linestyle='--', alpha=0.3)
    plt.axvline(0, color='gray', linestyle='--', alpha=0.3)
    plt.show()

# Ejemplo 1: f(x) = x², f'(x) = 2x
f1 = lambda x: x**2
f1_prime = lambda x: 2*x
plot_function_and_tangent(f1, f1_prime, x0=1.5)

# Ejemplo 2: f(x) = e^x, f'(x) = e^x
f2 = lambda x: np.exp(x)
f2_prime = lambda x: np.exp(x)
plot_function_and_tangent(f2, f2_prime, x0=0.5, x_range=(-2, 2))

# Ejemplo 3: f(x) = ln(x), f'(x) = 1/x (solo para x>0)
f3 = lambda x: np.log(x)
f3_prime = lambda x: 1/x
plot_function_and_tangent(f3, f3_prime, x0=1.5, x_range=(0.1, 3))

---
## 4. Derivada de la Función de Pérdida MSE

El **Error Cuadrático Medio (MSE)** para un modelo lineal $\hat{y} = wx + b$ con una muestra es:

$$L(w, b) = (wx + b - y)^2$$

Para un conjunto de $n$ muestras:

$$MSE(w, b) = \frac{1}{n} \sum_{i=1}^n (w x_i + b - y_i)^2$$

### 4.1. Derivada parcial con respecto al peso $w$

Aplicando la regla de la cadena:

$$\frac{\partial MSE}{\partial w} = \frac{1}{n} \sum_{i=1}^n 2(w x_i + b - y_i) \cdot x_i = \frac{2}{n} \sum_{i=1}^n x_i (\hat{y}_i - y_i)$$

A veces se omite el factor $2/n$ por conveniencia, usando:

$$\frac{\partial MSE}{\partial w} = \sum_{i=1}^n -x_i (y_i - \hat{y}_i)$$

In [ ]:
# Derivación simbólica del MSE con SymPy
w, b, x, y = sp.symbols('w b x y')

# Pérdida para una muestra
loss = (w*x + b - y)**2
print(f"Pérdida para una muestra: L = {loss}")

# Derivada parcial respecto a w
dL_dw = sp.diff(loss, w)
print(f"∂L/∂w = {dL_dw}")

# Derivada parcial respecto a b
dL_db = sp.diff(loss, b)
print(f"∂L/∂b = {dL_db}")

# Para un conjunto de datos, sumamos
n = 3
x_vals = [1, 2, 3]
y_vals = [2, 4, 6]
w_val = 1.5
b_val = 0.5

# Calculamos el gradiente numéricamente
grad_w = 0
grad_b = 0
for xi, yi in zip(x_vals, y_vals):
    y_pred = w_val * xi + b_val
    grad_w += -2 * xi * (yi - y_pred)  # -2 * x * (y - y_pred)
    grad_b += -2 * (yi - y_pred)       # -2 * (y - y_pred)

print(f"\n📊 Ejemplo numérico:")
print(f"Gradiente respecto a w: {grad_w:.4f}")
print(f"Gradiente respecto a b: {grad_b:.4f}")

### 4.2. Visualización del gradiente en el espacio de pesos

Para un problema simple con una sola muestra, visualicemos cómo la derivada nos indica la dirección de descenso.

In [ ]:
# Datos: una sola muestra (x=2, y=4)
x_single = 2
y_single = 4

# Función de pérdida en función de w (asumiendo b=0 para simplificar)
def loss_w(w):
    return (w * x_single - y_single)**2

def grad_w(w):
    return 2 * x_single * (w * x_single - y_single)

w_vals = np.linspace(0, 4, 100)
loss_vals = loss_w(w_vals)
grad_vals = grad_w(w_vals)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(w_vals, loss_vals, 'b-', linewidth=2)
plt.xlabel('w')
plt.ylabel('MSE')
plt.title('Función de Pérdida MSE vs w')
plt.grid(True)
plt.axvline(2, color='r', linestyle='--', label='w óptimo = 2')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(w_vals, grad_vals, 'r-', linewidth=2)
plt.xlabel('w')
plt.ylabel('∂L/∂w')
plt.title('Derivada (Gradiente) vs w')
plt.grid(True)
plt.axhline(0, color='black', linestyle='--')
plt.axvline(2, color='r', linestyle='--', label='w óptimo = 2')
plt.legend()

plt.suptitle('Relación entre la pérdida y su derivada', fontsize=14)
plt.tight_layout()
plt.show()

print("📌 Cuando w < 2, la derivada es negativa (indica que debemos aumentar w).")
print("📌 Cuando w > 2, la derivada es positiva (indica que debemos disminuir w).")
print("📌 En w=2, la derivada es cero (punto óptimo).")

---
## 5. Conexiones IA: Gradiente Descendente y Backpropagation

### 5.1. Gradiente Descendente

El algoritmo de **gradiente descendente** actualiza los pesos en la dirección opuesta al gradiente para minimizar la pérdida:

$$w := w - \eta \frac{\partial J}{\partial w}$$

donde $\eta$ es la **tasa de aprendizaje**.

### 5.2. Backpropagation

En redes neuronales, la **regla de la cadena** permite propagar el error desde la salida hasta las capas ocultas, calculando gradientes de forma eficiente.

### 5.3. Modelos Generativos (VAEs, GANs)

Estos modelos optimizan funciones objetivo derivables (ej. ELBO en VAEs, adversarial loss en GANs) usando gradientes.

In [ ]:
# Simulación simple de gradiente descendente para encontrar el w óptimo
w_current = 0.5
eta = 0.1
iterations = 20

w_history = [w_current]
loss_history = [loss_w(w_current)]

for i in range(iterations):
    grad = grad_w(w_current)
    w_current = w_current - eta * grad
    w_history.append(w_current)
    loss_history.append(loss_w(w_current))

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(w_vals, loss_vals, 'b-', alpha=0.5)
plt.scatter(w_history, loss_history, c='red', s=50, zorder=5)
plt.plot(w_history, loss_history, 'r--', alpha=0.5)
plt.xlabel('w')
plt.ylabel('MSE')
plt.title('Descenso del gradiente en la superficie de pérdida')
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(range(iterations+1), w_history, 'g-o')
plt.xlabel('Iteración')
plt.ylabel('w')
plt.title('Evolución del peso w')
plt.grid(True)
plt.axhline(2, color='r', linestyle='--', label='w óptimo')
plt.legend()

plt.suptitle('Gradiente Descendente en acción', fontsize=14)
plt.tight_layout()
plt.show()

print(f"Valor final de w: {w_current:.4f} (debería acercarse a 2)")

---
## 6. Derivadas de Funciones de Activación (Profundización)

Las funciones de activación deben ser diferenciables (o tener subgradientes) para permitir el backpropagation. Veamos sus derivadas.

In [ ]:
# Definimos funciones de activación y sus derivadas
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def sigmoid_prime(x):
    s = sigmoid(x)
    return s * (1 - s)

def tanh_prime(x):
    return 1 - np.tanh(x)**2

def relu(x):
    return np.maximum(0, x)

def relu_prime(x):
    return np.where(x > 0, 1, 0)

x_vals = np.linspace(-5, 5, 400)

plt.figure(figsize=(14, 10))

# Sigmoide
plt.subplot(2, 2, 1)
plt.plot(x_vals, sigmoid(x_vals), 'b-', label='σ(x)')
plt.plot(x_vals, sigmoid_prime(x_vals), 'r--', label='σ\'(x)')
plt.xlabel('x')
plt.ylabel('y')
plt.title('Sigmoide y su derivada')
plt.legend()
plt.grid(True)

# Tanh
plt.subplot(2, 2, 2)
plt.plot(x_vals, np.tanh(x_vals), 'b-', label='tanh(x)')
plt.plot(x_vals, tanh_prime(x_vals), 'r--', label='tanh\'(x)')
plt.xlabel('x')
plt.ylabel('y')
plt.title('Tanh y su derivada')
plt.legend()
plt.grid(True)

# ReLU
plt.subplot(2, 2, 3)
plt.plot(x_vals, relu(x_vals), 'b-', label='ReLU(x)')
plt.plot(x_vals, relu_prime(x_vals), 'r--', label='ReLU\'(x)')
plt.xlabel('x')
plt.ylabel('y')
plt.title('ReLU y su derivada (subgradiente)')
plt.legend()
plt.grid(True)

# Softplus (suave)
def softplus(x):
    return np.log(1 + np.exp(x))

def softplus_prime(x):
    return 1 / (1 + np.exp(-x))  # sigmoid

plt.subplot(2, 2, 4)
plt.plot(x_vals, softplus(x_vals), 'b-', label='Softplus(x)')
plt.plot(x_vals, softplus_prime(x_vals), 'r--', label='Softplus\'(x)')
plt.xlabel('x')
plt.ylabel('y')
plt.title('Softplus y su derivada (suave)')
plt.legend()
plt.grid(True)

plt.suptitle('Funciones de Activación y sus Derivadas', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

---
## Ejercicios para el Estudiante

1.  **Derivadas Simbólicas:** Usa SymPy para calcular la derivada de:
    $$f(x) = \frac{\sin(x)}{x}, \quad g(x) = e^{-x^2}, \quad h(x) = \ln(\tanh(x))$$

2.  **Interpretación Geométrica:** Para la función $f(x) = x^3 - 3x$, encuentra los puntos donde la derivada es cero. ¿Qué tipo de puntos críticos son? Visualiza.

3.  **Gradiente del MSE:** Deriva la expresión completa del gradiente del MSE para un modelo lineal con múltiples características. Escribe la expresión en forma matricial.

4.  **Comparación Numérica:** Implementa la derivada numérica (diferencias finitas) para $f(x) = e^{-x^2}$ y compárala con la derivada analítica en varios puntos. ¿Cómo afecta el tamaño de $h$?

5.  **Reflexión:** ¿Por qué es importante que las funciones de activación tengan derivadas que no se desvanezcan (o exploten) en el backpropagation? Investiga el problema del **gradiente desvaneciente**.

---
## Conclusiones de la Sesión

*   La **derivada** mide la tasa de cambio instantánea y geométricamente es la pendiente de la recta tangente.
*   Hemos calculado derivadas simbólicas de funciones simples y de activación usando **SymPy**.
*   La derivada de la función de pérdida **MSE** con respecto a los pesos nos indica cómo ajustarlos para minimizar el error.
*   El **gradiente descendente** usa estas derivadas para actualizar los pesos iterativamente.
*   En deep learning, la **regla de la cadena** permite propagar gradientes a través de la red (backpropagation).
*   Las funciones de activación deben ser diferenciables (o tener subgradientes) para que el entrenamiento sea posible.

¡La derivada es la brújula que guía a los modelos hacia el mínimo del error!